<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.4_Praktikum_Transfer_Learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.4: Praktikum Transfer Learning (Feature Extraction vs Fine-Tuning)
**Tujuan Pembelajaran:**
1. Membangun model Transfer Learning berbasis ResNet menggunakan mekanisme pembekuan bobot (`requires_grad = False`).
2. Menerapkan strategi optimasi *Differential Learning Rates* pada kelompok parameter jaringan.
3. Mensimulasikan klasifikasi 4 kelas defisiensi hara bibit kelapa sawit dengan dataset terbatas.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Simulasi Pretrained Backbone dan Pembekuan Bobot

In [ ]:
# Arsitektur Backbone yang Merepresentasikan Fitur Universal (Pretrained)
class PretrainedBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.stage2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
    def forward(self, x):
        return self.stage2(self.stage1(x))

# Model Transfer Learning dengan Custom Head
class AgroTransferNet(nn.Module):
    def __init__(self, num_classes=4, freeze_backbone=True):
        super().__init__()
        self.backbone = PretrainedBackbone()
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model_feat_extract = AgroTransferNet(freeze_backbone=True)
model_scratch = AgroTransferNet(freeze_backbone=False)

p_frozen = sum(p.numel() for p in model_feat_extract.parameters() if p.requires_grad)
p_total = sum(p.numel() for p in model_feat_extract.parameters())
print(f"Feature Extraction: {p_frozen:,} parameter aktif dari total {p_total:,} ({p_frozen/p_total*100:.2f}% aktif)")
print(f"From Scratch:       {sum(p.numel() for p in model_scratch.parameters() if p.requires_grad):,} parameter aktif (100% aktif)")

### Bagian 2: Sintesis Citra Defisiensi Unsur Hara Daun Sawit (N, K, Mg, B)
Membangun dataset terbatas: hanya 50 sampel per kelas (total 200 sampel) untuk mensimulasikan kelangkaan data riil.

In [ ]:
def generate_nutrient_deficiency_data(n_samples=200):
    X = np.zeros((n_samples, 3, 32, 32), dtype=np.float32)
    y = np.zeros(n_samples, dtype=np.int64)
    
    for i in range(n_samples):
        cls_idx = i % 4
        y[i] = cls_idx
        if cls_idx == 0:   # Defisiensi N (Kuning pucat merata)
            r, g, b = 0.70, 0.75, 0.20
        elif cls_idx == 1: # Defisiensi K (Bercak oranye)
            r, g, b = 0.80, 0.45, 0.10
        elif cls_idx == 2: # Defisiensi Mg (Jingga berjalur cerah)
            r, g, b = 0.65, 0.55, 0.15
        else:              # Defisiensi B (Hijau kusam berkerut)
            r, g, b = 0.25, 0.40, 0.20
            
        noise = np.random.normal(0, 0.05, (3, 32, 32))
        X[i, 0, :, :] = r + noise[0]
        X[i, 1, :, :] = g + noise[1]
        X[i, 2, :, :] = b + noise[2]
        
    X = np.clip(X, 0.0, 1.0)
    return torch.tensor(X), torch.tensor(y)

X_raw, y_raw = generate_nutrient_deficiency_data(200)
ds_full = TensorDataset(X_raw, y_raw)
train_ds, val_ds = torch.utils.data.random_split(ds_full, [140, 60])
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
print(f"Dataset Defisiensi Hara siap: {len(train_ds)} train, {len(val_ds)} val.")

### Bagian 3: Pelatihan dengan Differential Learning Rates

In [ ]:
# Model Fine-Tuning dengan Differential LR
model_finetune = AgroTransferNet(freeze_backbone=False)

# Kelompokkan parameter dengan laju pembelajaran berbeda
optimizer = optim.AdamW([
    {'params': model_finetune.backbone.parameters(), 'lr': 1e-4}, # Backbone: LR kecil
    {'params': model_finetune.head.parameters(),     'lr': 1e-2}  # Head:     LR besar
], weight_decay=1e-3)

criterion = nn.CrossEntropyLoss()
epochs = 12
losses, val_accs = [], []

for epoch in range(epochs):
    model_finetune.train()
    epoch_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = model_finetune(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * bx.size(0)
        
    loss_avg = epoch_loss / len(train_loader.dataset)
    losses.append(loss_avg)
    
    model_finetune.eval()
    correct = 0
    with torch.no_grad():
        for vx, vy in val_loader:
            preds = model_finetune(vx).argmax(dim=1)
            correct += (preds == vy).sum().item()
    acc = correct / len(val_loader.dataset)
    val_accs.append(acc)
    print(f"Epoch [{epoch+1:02d}/{epochs}] - Loss: {loss_avg:.4f} | Val Accuracy: {acc*100:.2f}%")

# Plot kurva pelatihan
plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
plt.plot(losses, 'r-o')
plt.title('Loss Pelatihan Fine-Tuning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(val_accs, 'b-s')
plt.title('Akurasi Validasi Defisiensi Hara')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.grid(True)
plt.tight_layout()
plt.savefig('praktikum_transfer_learning_12_4.png', dpi=200)
plt.close()
print("[OK] Grafik hasil pelatihan disimpan sebagai 'praktikum_transfer_learning_12_4.png'")